# 第 16 章 情報の非対称性: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/micro-basics-programming/16-information-asymmetry.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
# 逆選択（レモンの市場）
good_share = 0.5          # 良い品の割合
value_good = 100          # 良い品の、買い手にとっての価値
value_bad = 40            # 悪い品の、買い手にとっての価値
seller_reservation = 60   # 良い品の売り手の、最低価格
qualities = [20, 40, 60, 80, 100]  # 品質のリスト
premium = 1.5             # 買い手の評価の倍率（質 × 倍率 まで払う）
# シグナリング（教育）
wage_high = 100           # 高い能力の人の賃金
wage_low = 60             # 低い能力の人の賃金
cost_high = 10            # 教育 1 年あたりの費用（高い能力）
cost_low = 20             # 教育 1 年あたりの費用（低い能力。cost_high より大きく）
education = 3             # 調べる教育年数
e_max = 6                 # 図の教育年数の範囲
# モラルハザード（成果報酬）
value = 100               # 成功したときの、依頼人の価値
p_high = 0.8              # 努力したときの、成功の確率
p_low = 0.4               # 努力しないときの、成功の確率
effort_cost = 10          # 努力の費用
bonus = 25                # 成功したときの報酬
b_max = 60                # 図の報酬の範囲

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))
_loaded_lib = sys.modules.get("lib")
if _loaded_lib is not None and (
    Path(getattr(_loaded_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in tuple(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 逆選択

In [ ]:
from lib.chapter16.information import (
    expected_value_of_unknown_quality,
    good_quality_sells,
    unraveling,
)

willingness = expected_value_of_unknown_quality(good_share, value_good, value_bad)
print(f"質が分からない買い手の支払意思額: {willingness:.2f}")
sells = good_quality_sells(good_share, value_good, value_bad, seller_reservation)
print(f"良い品は売られるか（最低価格 {seller_reservation}）: {sells}")
print(f"評価の倍率 {premium} のとき、市場に残る質: {unraveling(qualities, premium)}")

## シグナリング

In [ ]:
from lib.chapter16.information import separating_education_range, signal_separates

low, high = separating_education_range(wage_high, wage_low, cost_high, cost_low)
print(f"区別できる教育年数の範囲: {low:.2f} 年から {high:.2f} 年")
separates = signal_separates(education, wage_high, wage_low, cost_high, cost_low)
print(f"教育 {education} 年で区別できるか: {separates}")

## モラルハザード

In [ ]:
from lib.chapter16.information import (
    agent_works,
    minimum_bonus,
    principal_profit_with_bonus,
    worth_paying_bonus,
)

print(f"努力させる、最低の成果報酬: {minimum_bonus(p_high, p_low, effort_cost):.2f}")
print(f"報酬 {bonus} で努力するか: {agent_works(bonus, p_high, p_low, effort_cost)}")
profit_effort = principal_profit_with_bonus(value, p_high, p_low, effort_cost)
print(f"依頼人の期待利潤（努力させる）: {profit_effort:.2f}、"
      f"努力なし: {p_low * value:.2f}")
worth = worth_paying_bonus(value, p_high, p_low, effort_cost)
print(f"努力させる価値があるか: {worth}")

## 図

In [ ]:
from lib.chapter16.plot import plot_signal

plot_signal(wage_high, wage_low, cost_high, cost_low, e_max)

In [ ]:
from lib.chapter16.plot import plot_bonus

plot_bonus(p_high, p_low, effort_cost, b_max)

## 試してみよう

- `good_share = 0.8` にすると、支払意思額は 88.0 で、`seller_reservation = 80` でも、
  良い品は売られます。`good_share = 0.2` では、支払意思額は 52.0 で、最低価格 60 の
  良い品も、売られません。**悪い品が多いほど、良い品が消えやすい**です
- `premium = 1.2` にすると、残るのは [20] だけ、`premium = 2.0` なら、全部が残ります。
  買い手の評価が高いほど、市場は崩れにくくなります
- `cost_low = 40` にすると、区別できる教育年数は 1.0 年から 4.0 年に広がります。
  `wage_high = 80` では、1.0 年から 2.0 年に狭まります。`cost_high = 15` では、
  2.0 年から約 2.67 年です
- `cost_low = 10`（`cost_high` と同じ）にすると、エラーになります。低い能力の人の
  ほうが、教育が苦しい、という前提が崩れるためです
- `effort_cost = 20` にすると、最低の報酬は 50.0 に上がり、依頼人の期待利潤は 40.0 で、
  努力させない場合と同じです（努力させる価値は、ありません）。`p_high = 0.9` にすると、
  最低の報酬は 20.0 に下がり、期待利潤は 72.0 に増えます